# Data Collection

In [ ]:
import os
import pandas as pd

NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = r"Data"

def load_data(NFL_YEAR):
    path = os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv")
    NFL = pd.read_csv(path)
    return NFL

for year in NFL_YEARS:
    df = load_data(year)
    print(f"{year} loaded — {len(df)} teams")

display(df.tail())

In [ ]:
# Global 60 / 20 / 20 Train / Validation / Test Split
# All models must import X_train_global, X_val_global, X_test_global, y_train_global, y_val_global, y_test_global rather than calling
# train_test_split independently, so every model trains and evaluates on identical data partitions

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Column indices (shared constants)
TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

NFL_TEAMS = ['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills',
             'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns',
             'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers',
             'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs',
             'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins',
             'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants',
             'New York Jets', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Francisco 49ers',
             'Seattle Seahawks', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders']
NFL_YEARS  = [str(year) for year in range(2010, 2026)]
DATA_FILEPATH = "Data"
DROP_COLS  = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV']


def _load_all_years():
    """Concatenate every year's data into one DataFrame."""
    frames = []
    for year in NFL_YEARS:
        path = os.path.join(DATA_FILEPATH, f"{year}.csv")
        raw  = pd.read_csv(path)
        # keep only rows that belong to a known team
        raw  = raw[raw.iloc[:, TEAM].isin(NFL_TEAMS)].copy()
        raw.columns = ['Team', 'W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD',
                       'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS']
        raw['Year'] = year
        frames.append(raw)
    combined = pd.concat(frames, ignore_index=True)
    for col in combined.columns:
        if col not in ('Team', 'Year'):
            combined[col] = pd.to_numeric(combined[col], errors='coerce')
    return combined.dropna()


# Build combined dataset
_all_data = _load_all_years()

# Features / target
_features = _all_data[['SoS', 'SRS', 'OSRS', 'DSRS']]
_target   = _all_data['W-L%']

# First split: 60 % train  |  40 % temp
X_train_global, _X_temp, y_train_global, _y_temp = train_test_split(
    _features.values, _target.values,
    test_size=0.40, random_state=42
)

# Second split: 50 % of temp → validation (20 % total)
#               50 % of temp → test       (20 % total)
X_val_global, X_test_global, y_val_global, y_test_global = train_test_split(
    _X_temp, _y_temp,
    test_size=0.50, random_state=42
)

print(f"Total samples  : {len(_features)}")
print(f"Train  (60 %)  : {len(X_train_global)}")
print(f"Val    (20 %)  : {len(X_val_global)}")
print(f"Test   (20 %)  : {len(X_test_global)}")

# Logistic Regression Model

## Greg Yonan

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Column indices
TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

NFL_TEAMS = ['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills',
             'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns',
             'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers',
             'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs',
             'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins',
             'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants',
             'New York Jets', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Francisco 49ers',
             'Seattle Seahawks', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders']
NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = "Data"
SCHEDULE_FILEPATH = "Schedule"
MODEL_FILEPATH = "Models/Logistic Regression"
PREDICTIONS_FILEPATH = "Predictions/Logistic Regression"

In [ ]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])

In [ ]:
def train_model():
    model = LinearRegression()
    model.fit(X_train_global, y_train_global)
    return model


In [ ]:
def save_model(NFL_YEAR, model):
    os.makedirs(MODEL_FILEPATH, exist_ok=True)
    with open(os.path.join(MODEL_FILEPATH, f"linear_regression_model_{NFL_YEAR}.pkl"), "wb") as file:
        pickle.dump(model, file)

In [ ]:
def make_predictions(NFL_YEAR):
    model_path = os.path.join(MODEL_FILEPATH, "linear_regression_model_global.pkl")
    with open(model_path, "rb") as file:
        model = pickle.load(file)

    schedule_df = pd.read_csv(os.path.join(SCHEDULE_FILEPATH, f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    try:
        season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
        team_stats = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM]) for _, row in season_data.iterrows()}
        team_stats = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}
    except FileNotFoundError:
        print(f"Data file for {NFL_YEAR} not found")
        return

    drop_cols = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV']
    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue
        winner_pred = model.predict(team_stats[winner].drop(columns=drop_cols))[0]
        loser_pred  = model.predict(team_stats[loser].drop(columns=drop_cols))[0]
        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner] += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df

In [ ]:
model = train_model()
save_model("global", model)
print("Model saved.")

# Evaluate on the global validation split
from sklearn.metrics import r2_score, mean_squared_error
y_pred_lr_val = model.predict(X_val_global)
print(f"R²: {r2_score(y_val_global, y_pred_lr_val):.4f}")
print(f"MSE: {mean_squared_error(y_val_global, y_pred_lr_val):.6f}")

# Run game-by-game predictions for each year using the global model
for year in NFL_YEARS:
    print(f"\nPredictions for {year}...")
    preds = make_predictions(year)
    if preds is not None:
        display(preds)


# Neural Network Model

## Dylan Shaffer

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

DATA_FILEPATH = "Data"
MODEL_FILEPATH = "Models/Neural Network"
PREDICTIONS_FILEPATH = "Predictions/Neural Network"

DROP_COLS = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV']

In [ ]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])

In [ ]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)  # single output: W-L%
    ])
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

In [ ]:
def train_model():
    # Compute normalisation stats from training data only
    mean = X_train_global.mean(axis=0)
    std  = X_train_global.std(axis=0)

    X_train_norm = (X_train_global - mean) / std
    X_val_norm   = (X_val_global   - mean) / std

    model = build_model(X_train_norm.shape[1])
    model.fit(
        X_train_norm, y_train_global,
        epochs=100, batch_size=8, verbose=0,
        validation_data=(X_val_norm, y_val_global)
    )
    return model, mean, std


In [ ]:
def save_model(NFL_YEAR, model, mean, std):
    os.makedirs(MODEL_FILEPATH, exist_ok=True)
    model.save(os.path.join(MODEL_FILEPATH, f"nn_model_{NFL_YEAR}.keras"))
    np.save(os.path.join(MODEL_FILEPATH, f"nn_mean_{NFL_YEAR}.npy"), mean)
    np.save(os.path.join(MODEL_FILEPATH, f"nn_std_{NFL_YEAR}.npy"),  std)

In [ ]:
def make_predictions(NFL_YEAR):
    model = keras.models.load_model(os.path.join(MODEL_FILEPATH, "nn_model_global.keras"))
    mean = np.load(os.path.join(MODEL_FILEPATH, "nn_mean_global.npy"))
    std = np.load(os.path.join(MODEL_FILEPATH, "nn_std_global.npy"))

    schedule_df = pd.read_csv(os.path.join("Schedule", f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
    team_stats  = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM]) for _, row in season_data.iterrows()}
    team_stats  = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue

        def predict_team(team):
            X = team_stats[team].drop(columns=DROP_COLS).values
            X = (X - mean) / std
            return model.predict(X, verbose=0)[0][0]

        winner_pred = predict_team(winner)
        loser_pred  = predict_team(loser)

        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner]  += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df

In [ ]:
model, mean, std = train_model()
save_model("global", model, mean, std)
print("Model saved.")

# Run game-by-game predictions for each year using the global model
for year in NFL_YEARS:
    print(f"\nPredictions for {year}...")
    preds = make_predictions(year)
    if preds is not None:
        display(preds)


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

# Normalize test set with the same stats used during training
X_test_norm = (X_test_global - mean) / std

# Raw predictions from the neural network
y_pred_nn = model.predict(X_test_norm, verbose=0).flatten()

r2  = r2_score(y_test_global, y_pred_nn)
mse = mean_squared_error(y_test_global, y_pred_nn)

# Percent error per sample (avoid divide-by-zero for 0.0 actual values)
with np.errstate(divide='ignore', invalid='ignore'):
    pct_err = np.where(
        y_test_global != 0,
        np.abs((y_test_global - y_pred_nn) / y_test_global) * 100,
        np.nan
    )
mean_pct_err = np.nanmean(pct_err)

print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.6f}")
print(f"Mean % Error: {mean_pct_err:.2f} %")


# Random Forest Model

## Simon Liedtke

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

RF_MODEL_FILEPATH = "Models/Random Forest"
RF_PREDICTIONS_FILEPATH = "Predictions/Random Forest"

SCHEDULE_FILEPATH = "Schedule"
DATA_FILEPATH = "Data"

TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

NFL_TEAMS = ['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills',
             'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns',
             'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers',
             'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs',
             'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins',
             'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants',
             'New York Jets', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Francisco 49ers',
             'Seattle Seahawks', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders']
NFL_YEARS = [str(year) for year in range(2010, 2026)]
DROP_COLS = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV']


In [ ]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])


In [ ]:
def train_model():
    # Hyperparameter search over the global training split
    param_grid = {
        'n_estimators':      [5000],
        'max_depth':         [None, 2, 20],
        'min_samples_split': [2, 20],
        'min_samples_leaf':  [2, 20],
        'bootstrap':         [True, False],
    }
    random_search = RandomizedSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid,
        n_iter=10,
        cv=3,
        random_state=42,
        n_jobs=-1
    )
    random_search.fit(X_train_global, y_train_global)
    print("Best hyperparameters:", random_search.best_params_)

    # Re-train with the best params (already done internally, but we expose it)
    model = random_search.best_estimator_
    return model


In [ ]:
def save_model(NFL_YEAR, model):
    os.makedirs(RF_MODEL_FILEPATH, exist_ok=True)
    with open(os.path.join(RF_MODEL_FILEPATH, f"rf_model_{NFL_YEAR}.pkl"), "wb") as f:
        pickle.dump(model, f)


In [ ]:
def make_predictions(NFL_YEAR):
    model_path = os.path.join(RF_MODEL_FILEPATH, "rf_model_global.pkl")
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    schedule_df = pd.read_csv(os.path.join(SCHEDULE_FILEPATH, f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    try:
        season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
        team_stats  = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM])
                       for _, row in season_data.iterrows()}
        team_stats  = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}
    except FileNotFoundError:
        print(f"Data file for {NFL_YEAR} not found")
        return

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue
        winner_pred = model.predict(team_stats[winner].drop(columns=DROP_COLS))[0]
        loser_pred  = model.predict(team_stats[loser].drop(columns=DROP_COLS))[0]
        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner]  += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(RF_PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(RF_PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df


In [ ]:
rf_model = train_model()
save_model("global", rf_model)
print("Model saved.")

# Evaluate on the global validation split
y_pred_rf_val = rf_model.predict(X_val_global)
print(f"R²:  {r2_score(y_val_global,  y_pred_rf_val):.4f}")
print(f"MSE: {mean_squared_error(y_val_global, y_pred_rf_val):.6f}")

# Run game-by-game predictions for each year using the global model
for year in NFL_YEARS:
    print(f"\nPredictions for {year}...")
    preds = make_predictions(year)
    if preds is not None:
        display(preds)


In [ ]:
# Evaluate on the global test split (held out until final evaluation)
y_pred_rf_test = rf_model.predict(X_test_global)

mse = mean_squared_error(y_test_global, y_pred_rf_test)
r2  = r2_score(y_test_global, y_pred_rf_test)
mae = mean_absolute_error(y_test_global, y_pred_rf_test)

# Percent error per sample
with np.errstate(divide='ignore', invalid='ignore'):
    pct_err = np.where(
        y_test_global != 0,
        np.abs((y_test_global - y_pred_rf_test) / y_test_global) * 100,
        np.nan
    )
mean_pct_err = np.nanmean(pct_err)

print("Random Forest — Test-Set Performance (global 20% hold-out)")
print(f"  R²            : {r2:.4f}")
print(f"  MSE           : {mse:.6f}")
print(f"  MAE           : {mae:.6f}")
print(f"  Mean % Error  : {mean_pct_err:.2f} %")


# Naive Bayes Model

## Dilen Patel

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    r2_score,
)

NB_MODEL_FILEPATH       = "Models/Naive Bayes"
NB_PREDICTIONS_FILEPATH = "Predictions/Naive Bayes"


In [ ]:
def wl_bucket_label(wlp):
    """Convert W-L% into one of 4 bucket classes."""
    if wlp < 0.250:
        return 0  # Very Poor
    elif wlp < 0.500:
        return 1  # Below Average
    elif wlp < 0.750:
        return 2  # Good
    else:
        return 3  # Elite


def bucket_name(bucket_id):
    return {0: "Very Poor", 1: "Below Average", 2: "Good", 3: "Elite"}.get(bucket_id, "Unknown")


def bucket_midpoint(bucket_id):
    """Return the midpoint W-L% for a given bucket (used as the numeric prediction)."""
    return {0: 0.125, 1: 0.375, 2: 0.625, 3: 0.875}.get(bucket_id, 0.0)


In [ ]:
# Apply bucket labels to the full dataset
_nb_data = _all_data.copy()
_nb_data["WL_bucket"] = _nb_data["W-L%"].apply(wl_bucket_label)

# ── Four-feature set (matches naive_bayes.py requirement) ──────────────────
NB_FEATURE_COLS = ["SoS", "SRS", "OSRS", "DSRS"]

# Re-index bucket labels to align with the global 60/20/20 split indices
_nb_y_all = _nb_data["WL_bucket"].values  # bucket targets, full dataset
_nb_X_all = _nb_data[NB_FEATURE_COLS].values

from sklearn.model_selection import train_test_split

# Mirror the same 60/20/20 random splits used by every other model
X_nb_train, _nb_X_temp, y_nb_train, _nb_y_temp = train_test_split(
    _nb_X_all, _nb_y_all, test_size=0.40, random_state=42
)
X_nb_val, X_nb_test, y_nb_val, y_nb_test = train_test_split(
    _nb_X_temp, _nb_y_temp, test_size=0.50, random_state=42
)

# Also keep the actual W-L% values aligned to the test indices for regression metrics
_nb_wlp_all = _nb_data["W-L%"].values
_, _wlp_temp, _, _ = train_test_split(
    _nb_wlp_all, _nb_y_all, test_size=0.40, random_state=42
)
_, wlp_nb_test, _, _ = train_test_split(
    _wlp_temp, _nb_y_temp, test_size=0.50, random_state=42
)

print(f"NB features : {NB_FEATURE_COLS}")
print(f"Train (60%) : {len(X_nb_train)}")
print(f"Val   (20%) : {len(X_nb_val)}")
print(f"Test  (20%) : {len(X_nb_test)}")


In [ ]:
def train_model():
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_nb_train)
    model = GaussianNB()
    model.fit(X_train_scaled, y_nb_train)
    return model, scaler


def save_model(model, scaler):
    os.makedirs(NB_MODEL_FILEPATH, exist_ok=True)
    joblib.dump(model,  os.path.join(NB_MODEL_FILEPATH, "nb_wlp_bucket.pkl"))
    joblib.dump(scaler, os.path.join(NB_MODEL_FILEPATH, "scaler_wlp_bucket.pkl"))


def evaluate_model(model, scaler):
    X_test_scaled = scaler.transform(X_nb_test)
    preds = model.predict(X_test_scaled)

    # ── Classification metrics ───────────────────────────────────────────────
    acc    = accuracy_score(y_nb_test, preds)
    report = classification_report(
        y_nb_test, preds,
        target_names=["Very Poor", "Below Average", "Good", "Elite"],
        zero_division=0,
    )
    cm = confusion_matrix(y_nb_test, preds)

    # ── W-L% regression metrics (bucket midpoint as numeric prediction) ──────
    predicted_wlp = np.array([bucket_midpoint(int(p)) for p in preds])
    actual_wlp    = wlp_nb_test.astype(float)

    mse = mean_squared_error(actual_wlp, predicted_wlp)
    r2  = r2_score(actual_wlp, predicted_wlp)

    with np.errstate(divide='ignore', invalid='ignore'):
        pct_errors = np.where(
            actual_wlp != 0,
            np.abs((actual_wlp - predicted_wlp) / actual_wlp) * 100,
            np.nan,
        )
    mean_pct_err = np.nanmean(pct_errors)

    # ── Print results (mirrors naive_bayes.py output) ────────────────────────
    print("Naive Bayes — Test-Set Performance (global 20 % hold-out)")
    print(f"  Accuracy       : {acc:.4f}")
    print(f"  MSE  (W-L%)    : {mse:.6f}")
    print(f"  R²   (W-L%)    : {r2:.6f}")
    print(f"  Mean % Error   : {mean_pct_err:.2f} %")
    print()
    print("Classification Report")
    print("-" * 40)
    print(report)
    print("Confusion Matrix")
    print("-" * 40)
    print(cm)

    # ── Build predictions DataFrame (mirrors naive_bayes.py predictions.csv) ─
    pred_rows = []
    for i in range(len(y_nb_test)):
        actual_bucket = int(y_nb_test[i])
        pred_bucket   = int(preds[i])
        actual_wl_val = float(actual_wlp[i])
        pred_wl_val   = float(predicted_wlp[i])
        abs_err       = abs(actual_wl_val - pred_wl_val)
        pct_err       = float(pct_errors[i]) if not np.isnan(pct_errors[i]) else None
        pred_rows.append({
            "Actual_WL_Percent"   : round(actual_wl_val, 3),
            "Predicted_WL_Percent": round(pred_wl_val, 3),
            "Absolute_Error"      : round(abs_err, 3),
            "Percent_Error"       : round(pct_err, 2) if pct_err is not None else None,
            "Actual_Bucket"       : bucket_name(actual_bucket),
            "Predicted_Bucket"    : bucket_name(pred_bucket),
            "Correct"             : int(actual_bucket == pred_bucket),
        })
    return pd.DataFrame(pred_rows)


def save_predictions(preds_df):
    os.makedirs(NB_PREDICTIONS_FILEPATH, exist_ok=True)
    preds_df.to_csv(
        os.path.join(NB_PREDICTIONS_FILEPATH, "predictions.csv"), index=False
    )


In [ ]:
nb_model, nb_scaler = train_model()
save_model(nb_model, nb_scaler)
print("Model saved.")

preds_df = evaluate_model(nb_model, nb_scaler)
save_predictions(preds_df)
print("\nPredictions saved.")
display(preds_df)
